# Extended Practice: COUNT and DISTINCT in SQL

This lab is an expanded version of the optional practice on `COUNT` and `DISTINCT`.  
You will move from basic row counts to business-style analytical questions using the **Chinook** music store database.

### Learning goals
By the end of this notebook you will be able to:
- Use `COUNT(*)` vs `COUNT(column)` and understand the difference
- Apply `COUNT(DISTINCT ...)` to measure uniqueness
- Combine counting with `WHERE`, `GROUP BY` and `HAVING`
- Answer realistic business questions about customers, invoices and catalog size
- Spot common pitfalls (NULLs, case sensitivity, wrong DISTINCT placement)

### Database
We use the classic **Chinook** SQLite database (digital music store).  
Main tables you will work with:
- `Customer` – 59 customers
- `Invoice` – 412 invoices
- `Track` – 3 503 tracks
- `Artist`, `Album`, `Genre`, `InvoiceLine`, etc.


## 1. Database Connection Setup

Before writing any queries you must:

1. Make sure the file **`chinook.db`** is in the **same folder** as this notebook  
   (or update the path in the next cell).
2. Run the connection cell below **once**.
3. Confirm you see the success message and the list of tables.

If you do not have the database yet, the setup cell will try to download it automatically.


In [ ]:
# ============================================================
# EXPLICIT DATABASE CONNECTION SETUP
# ============================================================
import os
import sqlite3
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the database file
# ------------------------------------------------------------
# Change this path if chinook.db is stored somewhere else.
DB_PATH = Path("chinook.db")          # same folder as the notebook
# DB_PATH = Path("/home/workdir/artifacts/chinook.db")  # absolute example

# ------------------------------------------------------------
# 2. Download the database if it is missing
# ------------------------------------------------------------
if not DB_PATH.exists():
    print(f"'{DB_PATH}' not found. Attempting to download Chinook sample database...")
    try:
        import urllib.request
        url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
        urllib.request.urlretrieve(url, DB_PATH)
        print(f"Download successful → {DB_PATH.resolve()}")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not find or download chinook.db.\n"
            f"Please place the file next to this notebook or update DB_PATH.\n"
            f"Original error: {e}"
        )
else:
    print(f"Database found: {DB_PATH.resolve()}")

# ------------------------------------------------------------
# 3. Create the connection
# ------------------------------------------------------------
# sqlite3.connect() opens (or creates) a connection to the SQLite file.
# check_same_thread=False is sometimes needed in Jupyter environments.
conn = sqlite3.connect(DB_PATH, check_same_thread=False)

# Optional but useful: make SQLite return rows that behave like dictionaries
conn.row_factory = sqlite3.Row

print("Connection object created:", conn)
print("SQLite version:", sqlite3.sqlite_version)

# ------------------------------------------------------------
# 4. Helper function – run any SQL and get a pandas DataFrame
# ------------------------------------------------------------
def run(sql: str, params=None) -> pd.DataFrame:
    """
    Execute a SQL query against the open connection and return
    the result as a clean pandas DataFrame.
    
    Parameters
    ----------
    sql : str
        The SQL statement to execute.
    params : tuple or list, optional
        Parameters for parameterized queries.
    """
    return pd.read_sql_query(sql, conn, params=params)

# ------------------------------------------------------------
# 5. Verify the connection works
# ------------------------------------------------------------
tables = run("""
    SELECT name 
    FROM sqlite_master 
    WHERE type = 'table' 
    ORDER BY name
""")

print("\nTables available in the database:")
print(tables.to_string(index=False))

# Quick sanity check
customer_count = run("SELECT COUNT(*) AS n FROM Customer")["n"].iloc[0]
print(f"\nSanity check → Customer table has {customer_count} rows.")
print("Setup complete. You can now use run(\"\"\" SELECT ... \"\"\") in any cell.")


### How to use the connection in the rest of the notebook

Every practice cell already contains a call to the helper:

```python
run("""
SELECT ...
""")
```

You only need to write the SQL between the triple quotes.  
If you prefer the classic cursor style you can also do:

```python
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM Customer")
print(cursor.fetchone())
```

Both styles talk to the **same** open connection (`conn`).


---
## Quick Schema Reference

| Table       | Key columns                                      | Notes                          |
|-------------|--------------------------------------------------|--------------------------------|
| Customer    | CustomerId, FirstName, LastName, Country, City, SupportRepId | 59 rows                    |
| Invoice     | InvoiceId, CustomerId, InvoiceDate, Total, BillingCountry | 412 rows                   |
| Track       | TrackId, Name, AlbumId, GenreId, Composer, Milliseconds | 3 503 rows                 |
| Artist      | ArtistId, Name                                   | 275 rows                       |
| Genre       | GenreId, Name                                    | 25 rows                        |
| InvoiceLine | InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity | line items                 |


---
# Section 1 – Basic COUNT

`COUNT(*)` counts **rows**.  
`COUNT(column)` counts **non-NULL values** in that column.


### Exercise 1.1 – Total number of customers

The sales team wants a single number: how many customers are in the database?

**Task**  
Write a query that returns the total number of rows in `Customer` under the alias `num_customers`.


In [ ]:
# Your query here
run("""
    
""")


**Expected output**
```
   num_customers
0             59
```


### Exercise 1.2 – COUNT(*) vs COUNT(column)

Some customers have no company name (`Company` is NULL).

**Task**  
1. Count all rows (`COUNT(*)`).  
2. Count non-NULL values in the `Company` column.  
Return both numbers in one result set with clear aliases.


In [ ]:
# Your query here
run("""
    
""")


**Expected output** (exact numbers may vary slightly by Chinook version, but the pattern is the same)
```
   total_customers  customers_with_company
0               59                      10
```
> Insight: `COUNT(Company)` is much smaller → most customers are individuals, not companies.


---
# Section 2 – COUNT DISTINCT

`COUNT(DISTINCT column)` tells you **how many unique values** exist.


### Exercise 2.1 – Unique customers (safety check)

To confirm there are no duplicate customer records, compare total rows with distinct `CustomerId`.

**Task**  
Return both counts side-by-side:
- `total_rows`
- `unique_customer_ids`


In [ ]:
# Your query here
run("""
    
""")


**Expected**
```
   total_rows  unique_customer_ids
0          59                   59
```


### Exercise 2.2 – How many countries do we serve?

**Task**  
Count the number of distinct countries in the `Customer` table.  
Alias: `num_countries`.


In [ ]:
# Your query here
run("""
    
""")


**Expected**
```
   num_countries
0             24
```


### Exercise 2.3 – Unique cities and unique city+country combinations

Sometimes the same city name exists in different countries.

**Task**  
Return three numbers in one query:
1. Distinct cities (`num_cities`)
2. Distinct countries (`num_countries`)
3. Distinct combinations of City + Country (`num_city_country`)


In [ ]:
# Your query here
run("""
    
""")


**Expected**
```
   num_cities  num_countries  num_city_country
0          53             24                53
```
(In this dataset every city name is unique across countries, so the last two numbers match.)


---
# Section 3 – Filtering with WHERE + COUNT

You often need counts of a **subset** of the data.


### Exercise 3.1 – Customers in the USA and Brazil

**Task**  
Write two separate counts (or one query with conditional aggregation) that return:
- Number of customers in the United States
- Number of customers in Brazil


In [ ]:
# Option A – two simple queries
run("""
    
""")

# Option B – single query with CASE (bonus)
run("""
    
""")


### Exercise 3.2 – High-value invoices

**Task**  
How many invoices have a total greater than or equal to $10?
Alias: `high_value_invoices`.


In [ ]:
# Your query here
run("""
    
""")


### Exercise 3.3 – Distinct countries that placed high-value orders

**Task**  
How many **distinct** billing countries appear on invoices where `Total >= 10`?


In [ ]:
# Your query here
run("""
    
""")


---
# Section 4 – GROUP BY + COUNT (the real power)

`GROUP BY` + `COUNT` answers “how many of each …?” questions.


### Exercise 4.1 – Customers per country

**Task**  
List every country together with the number of customers who live there.  
Order by the count descending (most customers first).  
Columns: `Country`, `num_customers`.


In [ ]:
# Your query here
run("""
    
""")


**Expected (top rows)**
```
         Country  num_customers
0            USA             13
1         Canada             8
2         France             5
3         Brazil             5
...
```


### Exercise 4.2 – Number of invoices per customer

**Task**  
For every customer, show:
- CustomerId
- FirstName + LastName (concatenate them)
- Number of invoices they have

Order by number of invoices descending. Limit to the top 10.


In [ ]:
# Your query here
run("""
    
""")


### Exercise 4.3 – Tracks per genre

**Task**  
Show each genre name and how many tracks belong to it.  
Order by track count descending.


In [ ]:
# Your query here
run("""
    
""")


---
# Section 5 – HAVING (filter after aggregation)

`WHERE` filters rows **before** grouping.  
`HAVING` filters groups **after** aggregation.


### Exercise 5.1 – Countries with more than 3 customers

**Task**  
Using the customers-per-country query, keep only countries that have **more than 3** customers.


In [ ]:
# Your query here
run("""
    
""")


### Exercise 5.2 – Customers who bought more than 6 times

**Task**  
Find customers who have more than 6 invoices.  
Return CustomerId, full name and invoice count.


In [ ]:
# Your query here
run("""
    
""")


### Exercise 5.3 – Genres with at least 100 tracks

**Task**  
Which genres contain 100 or more tracks?


In [ ]:
# Your query here
run("""
    
""")


---
# Section 6 – Combining COUNT DISTINCT with GROUP BY

Sometimes you need uniqueness **inside** each group.


### Exercise 6.1 – Distinct customers per country who have placed an order

Not every customer may have an invoice (in Chinook they all do, but the pattern is useful).

**Task**  
For each billing country, count how many **distinct** customers appear in the `Invoice` table.  
Order by that count descending.


In [ ]:
# Your query here
run("""
    
""")


### Exercise 6.2 – Number of distinct tracks sold per genre

**Task**  
Join `InvoiceLine` → `Track` → `Genre` and count, for each genre, how many **unique tracks** have ever been sold (appeared on at least one invoice line).


In [ ]:
# Your query here
run("""
    
""")


---
# Section 7 – Mini Business Challenge

Answer the following questions with clean, readable queries.  
These are the kinds of questions you will see in interviews or on the job.


### Challenge 7.1
What is the average number of invoices per customer?  
(Hint: total invoices ÷ total customers. You can do it with two counts or a subquery.)


In [ ]:
# Your query here
run("""
    
""")


### Challenge 7.2
Which support representative (SupportRepId) has the most customers?  
Show the SupportRepId and the count.


In [ ]:
# Your query here
run("""
    
""")


### Challenge 7.3
How many artists have more than one album?


In [ ]:
# Your query here
run("""
    
""")


### Challenge 7.4
List the top 5 countries by total invoice amount (`SUM(Total)`), together with the number of invoices and the number of distinct customers from that country.


In [ ]:
# Your query here
run("""
    
""")


---
# SOLUTIONS

Try the exercises first! Scroll down only when you want to check your answers.


### Solution 1.1
```sql
SELECT COUNT(*) AS num_customers
FROM Customer;
```


### Solution 1.2
```sql
SELECT 
    COUNT(*)          AS total_customers,
    COUNT(Company)    AS customers_with_company
FROM Customer;
```


### Solution 2.1
```sql
SELECT 
    COUNT(*)                    AS total_rows,
    COUNT(DISTINCT CustomerId)  AS unique_customer_ids
FROM Customer;
```


### Solution 2.2
```sql
SELECT COUNT(DISTINCT Country) AS num_countries
FROM Customer;
```


### Solution 2.3
```sql
SELECT 
    COUNT(DISTINCT City)                    AS num_cities,
    COUNT(DISTINCT Country)                 AS num_countries,
    COUNT(DISTINCT City || '-' || Country)  AS num_city_country
FROM Customer;
```


### Solution 3.1
```sql
-- Simple version
SELECT COUNT(*) AS usa_customers
FROM Customer
WHERE Country = 'USA';

SELECT COUNT(*) AS brazil_customers
FROM Customer
WHERE Country = 'Brazil';

-- Single query version
SELECT 
    SUM(CASE WHEN Country = 'USA'    THEN 1 ELSE 0 END) AS usa_customers,
    SUM(CASE WHEN Country = 'Brazil' THEN 1 ELSE 0 END) AS brazil_customers
FROM Customer;
```


### Solution 3.2
```sql
SELECT COUNT(*) AS high_value_invoices
FROM Invoice
WHERE Total >= 10;
```


### Solution 3.3
```sql
SELECT COUNT(DISTINCT BillingCountry) AS num_countries
FROM Invoice
WHERE Total >= 10;
```


### Solution 4.1
```sql
SELECT 
    Country,
    COUNT(*) AS num_customers
FROM Customer
GROUP BY Country
ORDER BY num_customers DESC;
```


### Solution 4.2
```sql
SELECT 
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS full_name,
    COUNT(i.InvoiceId) AS num_invoices
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, full_name
ORDER BY num_invoices DESC
LIMIT 10;
```


### Solution 4.3
```sql
SELECT 
    g.Name AS genre,
    COUNT(t.TrackId) AS num_tracks
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.Name
ORDER BY num_tracks DESC;
```


### Solution 5.1
```sql
SELECT 
    Country,
    COUNT(*) AS num_customers
FROM Customer
GROUP BY Country
HAVING COUNT(*) > 3
ORDER BY num_customers DESC;
```


### Solution 5.2
```sql
SELECT 
    c.CustomerId,
    c.FirstName || ' ' || c.LastName AS full_name,
    COUNT(i.InvoiceId) AS num_invoices
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, full_name
HAVING COUNT(i.InvoiceId) > 6
ORDER BY num_invoices DESC;
```


### Solution 5.3
```sql
SELECT 
    g.Name AS genre,
    COUNT(t.TrackId) AS num_tracks
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.Name
HAVING COUNT(t.TrackId) >= 100
ORDER BY num_tracks DESC;
```


### Solution 6.1
```sql
SELECT 
    BillingCountry,
    COUNT(DISTINCT CustomerId) AS num_customers
FROM Invoice
GROUP BY BillingCountry
ORDER BY num_customers DESC;
```


### Solution 6.2
```sql
SELECT 
    g.Name AS genre,
    COUNT(DISTINCT il.TrackId) AS distinct_tracks_sold
FROM InvoiceLine il
JOIN Track t ON il.TrackId = t.TrackId
JOIN Genre g ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY distinct_tracks_sold DESC;
```


### Solution 7.1
```sql
SELECT 
    ROUND(
        (SELECT COUNT(*) FROM Invoice) * 1.0 / 
        (SELECT COUNT(*) FROM Customer), 
        2
    ) AS avg_invoices_per_customer;
```


### Solution 7.2
```sql
SELECT 
    SupportRepId,
    COUNT(*) AS num_customers
FROM Customer
GROUP BY SupportRepId
ORDER BY num_customers DESC
LIMIT 1;
```


### Solution 7.3
```sql
SELECT COUNT(*) AS artists_with_multiple_albums
FROM (
    SELECT ArtistId
    FROM Album
    GROUP BY ArtistId
    HAVING COUNT(*) > 1
);
```


### Solution 7.4
```sql
SELECT 
    BillingCountry,
    ROUND(SUM(Total), 2)          AS total_revenue,
    COUNT(*)                      AS num_invoices,
    COUNT(DISTINCT CustomerId)    AS num_customers
FROM Invoice
GROUP BY BillingCountry
ORDER BY total_revenue DESC
LIMIT 5;
```


---
## Key Takeaways

| Concept | Pattern | When to use |
|---------|---------|-------------|
| `COUNT(*)` | Count every row | Total records, even if some columns are NULL |
| `COUNT(col)` | Count non-NULL values | When missing data matters |
| `COUNT(DISTINCT col)` | Unique values | Cardinality / uniqueness checks |
| `GROUP BY` + `COUNT` | Count per group | “How many X per Y?” |
| `HAVING COUNT(...)` | Filter groups | “Only groups that meet a threshold” |
| `COUNT(DISTINCT ...)` inside `GROUP BY` | Unique items inside each group | “How many unique customers per country?” |

### Common pitfalls to avoid
1. Putting `DISTINCT` in the wrong place (`SELECT DISTINCT COUNT(...)` is almost never what you want).
2. Forgetting that `COUNT(column)` ignores NULLs.
3. Using `WHERE` when you actually need `HAVING` (or vice-versa).
4. Not aliasing calculated columns → hard-to-read result sets.

---
**Next steps**  
- Add more joins (Playlist, MediaType, Employee).  
- Explore window functions: `COUNT(*) OVER (PARTITION BY Country)`.  
- Practice writing the same logic with CTEs for readability.
